# Notebook 07 — Dashboard Ejecutivo (Capa de Datos)

**Proyecto:** Sistema de Predicción de Demanda Hospitalaria — EsSalud
**Curso:** Big Data DD283 (2026-1) · **Grupo 3** · Junior Ortiz
**Semana 8 del cronograma — Evaluación Final**

---

## Criterio de éxito (README §8, fila S8 y §15)

| Requisito | Meta | Fuente |
|-----------|------|--------|
| Dashboard ejecutivo | Mínimo **7 KPIs** hospitalarios funcionales | §15 |
| Mapa de Lima | Demanda proyectada por establecimiento | §15 |
| Vistas gerenciales | Director · Gerente de Red · Epidemiólogo | §4 |

## Contenido

1. Inventario de datos para el dashboard
2. Inspección de columnas y campos para proxies de KPI
3. KPIs de gestión hospitalaria (los 8 del README §7.4)
4. Datos Vista 1 — Director (ocupación vs predicción)
5. Datos Vista 2 — Gerente de Red (mapa de Lima)
6. Datos Vista 3 — Epidemiólogo (correlación y vigilancia)
7. Verificación de insumos del dashboard
8. Conclusiones

## Arquitectura

Este notebook es la **capa de datos**: calcula y persiste en `data/gold/`
los insumos que consume el dashboard. La **capa de presentación** es una app
Streamlit (`src/dashboard.py`) que lee esos archivos y renderiza las 3 vistas.

> **Ejecución del dashboard:** local, con `streamlit run src/dashboard.py`.
> No se despliega en Streamlit Cloud porque los datos (capa Gold, 500K
> atenciones) están excluidos del repositorio por `.gitignore` y superan el
> límite de GitHub. La demo en vivo se realiza en local, como indica el README §8.

In [1]:
# ============================================================
#  1. INVENTARIO DE DATOS PARA EL DASHBOARD
# ============================================================
# Antes de calcular KPIs se verifica que insumos existen. El README (7.4)
# define 8 KPIs, pero varios requieren campos que el dataset sintetico
# puede no tener (egresos, fallecidos, horas-medico). Se determina cuales
# son CALCULABLES y cuales deben declararse como no disponibles.

import os
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# --- Localizar la raiz del proyecto ---
aqui = Path(os.getcwd())
raiz = next((p for p in [aqui, *aqui.parents] if (p / "data" / "gold").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("No se encontro 'data/gold'. Directorio: " + str(aqui))
RUTA_GOLD = raiz / "data" / "gold"
RUTA_SILVER = raiz / "data" / "silver" / "atenciones"
print("Raiz del proyecto:", raiz, "\n")

# --- Tablas Gold disponibles ---
print("Tablas en la capa Gold:")
for p in sorted(RUTA_GOLD.iterdir()):
    print(f"   {p.name}")
print()

# --- Columnas de la capa Plata (fuente de la mayoria de KPIs) ---
his = pd.read_parquet(RUTA_SILVER)
print(f"Capa Plata: {len(his):,} registros, {his.shape[1]} columnas\n")
print("Columnas disponibles:")
print(f"   {sorted(his.columns.tolist())}\n")

# --- Cruce contra los 8 KPIs del README (7.4) ---
KPIS_README = {
    "Tasa de Ocupacion":       ["camas_ocupadas_dia", "camas_disponibles_dia"],
    "Indice de Rotacion":      ["egresos", "camas_disponibles_dia"],
    "Promedio de Estadia":     ["dias_estancia", "egresos"],
    "Rendimiento Medico":      ["consultas", "horas_medico"],
    "Productividad Cama":      ["camas_ocupadas_dia", "camas_disponibles_dia"],
    "Lista Espera (dias)":     ["lista_espera_dias"],
    "Mortalidad Intrahosp.":   ["fallecidos", "egresos"],
    "IRAB":                    ["rechazos", "atenciones_totales"],
}

cols = set(his.columns)
print("Evaluacion de calculabilidad de los 8 KPIs del README:\n")
filas = []
for kpi, requiere in KPIS_README.items():
    faltan = [c for c in requiere if c not in cols]
    filas.append({
        "KPI": kpi,
        "campos_requeridos": ", ".join(requiere),
        "faltan": ", ".join(faltan) if faltan else "-",
        "estado": "CALCULABLE" if not faltan else "PROXY/NO DISPONIBLE",
    })
inv = pd.DataFrame(filas)
display(inv)

n_ok = int((inv["estado"] == "CALCULABLE").sum())
print(f"\nKPIs directamente calculables: {n_ok} de 8")
print(f"Meta del README: minimo 7 KPIs funcionales")

Raiz del proyecto: c:\Users\Home\Desktop\Grupo3\bigdata-g3-demanda-hospitalaria 

Tablas en la capa Gold:
   brechas_oferta_demanda_por_establecimiento
   correlacion_clima_enfermedad
   demanda_diaria
   demanda_semanal_por_especialidad
   kpis_dashboard.parquet
   prediccion_proximas_4_semanas.parquet
   vista1_director.parquet
   vista2_mapa.parquet
   vista3_comparativo_anual.parquet
   vista3_correlacion.parquet
   vista3_series_dengue.parquet

Capa Plata: 500,000 registros, 40 columnas

Columnas disponibles:
   ['alerta_epidemiologica', 'atencion_id', 'camas_disponibles_dia', 'camas_ocupadas_dia', 'casos_dengue', 'casos_dengue_semana', 'casos_influenza', 'cie10_capitulo', 'costo_atencion_soles', 'diagnostico_cie10', 'diagnostico_grupo', 'diagnostico_nombre', 'dias_estancia', 'especialidad', 'establecimiento_id', 'establecimiento_nombre', 'fecha_atencion', 'grupo_etario', 'indice_humedad', 'lista_espera_dias', 'medico_id', 'paciente_distrito', 'paciente_edad', 'paciente_grupo_etar

,KPI,campos_requeridos,faltan,estado
0,Tasa de Ocupacion,"camas_ocupadas_dia, camas_disponibles_dia",-,CALCULABLE
1,Indice de Rotacion,"egresos, camas_disponibles_dia",egresos,PROXY/NO DISPONIBLE
2,Promedio de Estadia,"dias_estancia, egresos",egresos,PROXY/NO DISPONIBLE
3,Rendimiento Medico,"consultas, horas_medico","consultas, horas_medico",PROXY/NO DISPONIBLE
4,Productividad Cama,"camas_ocupadas_dia, camas_disponibles_dia",-,CALCULABLE
5,Lista Espera (dias),lista_espera_dias,-,CALCULABLE
6,Mortalidad Intrahosp.,"fallecidos, egresos","fallecidos, egresos",PROXY/NO DISPONIBLE
7,IRAB,"rechazos, atenciones_totales","rechazos, atenciones_totales",PROXY/NO DISPONIBLE



KPIs directamente calculables: 3 de 8
Meta del README: minimo 7 KPIs funcionales


In [2]:
# ============================================================
#  2. INSPECCION DE COLUMNAS Y CAMPOS PARA PROXIES DE KPI
# ============================================================
# --- Verificar campos disponibles para construir los proxies de KPI ---
print("TODAS las columnas de la capa Plata:\n")
for i, c in enumerate(sorted(his.columns)):
    print(f"   {c:28s}", end="")
    if (i + 1) % 3 == 0:
        print()
print("\n")

# Campos clave para los proxies
print("Valores de 'tipo_atencion':")
print(f"   {his['tipo_atencion'].value_counts().to_dict()}\n")

print("Estadisticas de 'dias_estancia':")
print(f"   {his['dias_estancia'].describe().round(2).to_dict()}\n")

print("Rango de 'lista_espera_dias':")
print(f"   min={his['lista_espera_dias'].min()}  "
      f"max={his['lista_espera_dias'].max()}  "
      f"media={his['lista_espera_dias'].mean():.1f}\n")

print("Medicos unicos:", his["medico_id"].nunique())
print("Establecimientos:", his["establecimiento_nombre"].nunique())

# ¿Hay alguna señal de egreso/alta/fallecido en 'resultado_atencion'?
if "resultado_atencion" in his.columns:
    print("\nValores de 'resultado_atencion':")
    print(f"   {his['resultado_atencion'].value_counts().to_dict()}")

TODAS las columnas de la capa Plata:

   alerta_epidemiologica          atencion_id                    camas_disponibles_dia       
   camas_ocupadas_dia             casos_dengue                   casos_dengue_semana         
   casos_influenza                cie10_capitulo                 costo_atencion_soles        
   diagnostico_cie10              diagnostico_grupo              diagnostico_nombre          
   dias_estancia                  especialidad                   establecimiento_id          
   establecimiento_nombre         fecha_atencion                 grupo_etario                
   indice_humedad                 lista_espera_dias              medico_id                   
   paciente_distrito              paciente_edad                  paciente_grupo_etario       
   paciente_sexo                  precip_lag14                   precip_lag21                
   precip_lag7                    precipitacion_mm               red_asistencial             
   resultado_atencion 

In [3]:
# ============================================================
#  3. KPIs DE GESTION HOSPITALARIA (README seccion 7.4)
# ============================================================
# Se calculan los 8 KPIs definidos en el README. Fuente: capa Plata y la
# tabla Gold de brechas. Precisiones metodologicas declaradas:
#
#  - Promedio de Estadia: solo sobre hospitalizaciones (consulta y
#    emergencia tienen dias_estancia = 0; promediar las 500K daria 1.3).
#  - Egresos: toda atencion con desenlace en resultado_atencion.
#  - Rendimiento Medico: usa 'medicos_activos' de la tabla de oferta
#    (32-52 por sede), NO los 9,000 medico_id del HIS, que son
#    identificadores sinteticos repartidos por igual entre los 3
#    establecimientos (quinto caso de campo incoherente entre fuentes).
#  - IRAB: se aproxima con el deficit entre consultas ejecutadas y
#    programadas de la tabla de brechas (proxy declarado).
#  - Mortalidad: los desenlaces son aleatorios en el dataset sintetico;
#    el KPI es correcto como calculo pero no clinicamente interpretable.

JORNADA_SEMANAL_MEDICO = 30   # horas asistenciales/semana (supuesto declarado)

brechas = pd.read_parquet(RUTA_GOLD / "brechas_oferta_demanda_por_establecimiento")

# --- Insumos base ---
n_atenciones = len(his)
n_semanas = his["semana"].nunique()
hosp = his[his["tipo_atencion"] == "hospitalizacion"]
consultas = his[his["tipo_atencion"] == "consulta_externa"]

egresos = his["resultado_atencion"].isin(
    ["alta", "transferido", "fallecido", "fuga"]).sum()
fallecidos = (his["resultado_atencion"] == "fallecido").sum()

camas_tot = his.groupby("establecimiento_nombre")["camas_disponibles_dia"].first().sum()
camas_ocup_prom = his["camas_ocupadas_dia"].mean()

# medicos activos reales (de la fuente de oferta, no del HIS)
medicos_activos = brechas.groupby("establecimiento_nombre")["medicos_activos"].first().sum()

kpis = []

# 1. Tasa de Ocupacion
tasa_ocup = his["tasa_ocupacion"].mean() * 100
kpis.append(("Tasa de Ocupacion", f"{tasa_ocup:.1f}%", "85-90%",
             85 <= tasa_ocup <= 90, "real"))

# 2. Indice de Rotacion = egresos / camas / anios
rotacion = egresos / camas_tot / 3
kpis.append(("Indice de Rotacion", f"{rotacion:.1f} egresos/cama/anio",
             "> 40", rotacion > 40, "real"))

# 3. Promedio de Estadia (solo hospitalizacion)
estadia = hosp["dias_estancia"].mean()
kpis.append(("Promedio de Estadia", f"{estadia:.1f} dias", "< 5",
             estadia < 5, "real (solo hosp.)"))

# 4. Rendimiento Medico = consultas / (medicos_activos x jornada x semanas)
horas_medico = medicos_activos * JORNADA_SEMANAL_MEDICO * n_semanas
rendimiento = len(consultas) / horas_medico
kpis.append(("Rendimiento Medico", f"{rendimiento:.2f} consultas/hora",
             "> 4", rendimiento > 4, f"real ({medicos_activos} medicos activos)"))

# 5. Productividad Cama
productividad = camas_ocup_prom / camas_tot
kpis.append(("Productividad Cama", f"{productividad:.2f}", "> 0.85",
             productividad > 0.85, "real"))

# 6. Lista de Espera
espera = his["lista_espera_dias"].mean()
kpis.append(("Lista de Espera", f"{espera:.1f} dias", "< 30",
             espera < 30, "real"))

# 7. Mortalidad Intrahospitalaria = fallecidos / egresos x 1000
mortalidad = fallecidos / egresos * 1000
kpis.append(("Mortalidad Intrahosp.", f"{mortalidad:.1f} x1000 egresos",
             "< 25", mortalidad < 25, "real (desenlace aleatorio)"))

# 8. IRAB = deficit ejecutadas vs programadas [PROXY]
deficit = (brechas["consultas_ejecutadas"] < brechas["consultas_programadas"])
irab = deficit.mean() * 100
kpis.append(("IRAB", f"{irab:.1f}%", "< 2",
             irab < 2, "PROXY: ejecutadas < programadas"))

# --- Ensamblar ---
kpi_df = pd.DataFrame(kpis, columns=["KPI", "valor", "meta_essalud",
                                     "en_meta", "fuente"])
kpi_df["estado"] = kpi_df["en_meta"].map({True: "EN META", False: "FUERA DE META"})
kpi_df = kpi_df.drop(columns="en_meta")

print(f"KPIs de gestion hospitalaria — {n_atenciones:,} atenciones, "
      f"{n_semanas} semanas\n")
display(kpi_df)

n_meta = int((kpi_df["estado"] == "EN META").sum())
n_real = int(kpi_df["fuente"].str.startswith("real").sum())
print(f"\nKPIs calculados     : {len(kpi_df)} de 8")
print(f"Sobre datos reales  : {n_real}  |  con proxy declarado: {len(kpi_df)-n_real}")
print(f"Dentro de meta      : {n_meta}")
print(f"\nCriterio README (minimo 7 KPIs funcionales): "
      f"{'CUMPLE' if len(kpi_df) >= 7 else 'NO CUMPLE'}")

# Guardar para el dashboard
kpi_df.to_parquet(RUTA_GOLD / "kpis_dashboard.parquet", index=False)
print(f"\nGuardado: {RUTA_GOLD / 'kpis_dashboard.parquet'}")

KPIs de gestion hospitalaria — 500,000 atenciones, 158 semanas



,KPI,valor,meta_essalud,fuente,estado
0,Tasa de Ocupacion,92.4%,85-90%,real,FUERA DE META
1,Indice de Rotacion,123.5 egresos/cama/anio,> 40,real,EN META
2,Promedio de Estadia,6.5 dias,< 5,real (solo hosp.),FUERA DE META
3,Rendimiento Medico,0.41 consultas/hora,> 4,real (121 medicos activos),FUERA DE META
4,Productividad Cama,0.31,> 0.85,real,FUERA DE META
5,Lista de Espera,15.1 dias,< 30,real,EN META
6,Mortalidad Intrahosp.,29.7 x1000 egresos,< 25,real (desenlace aleatorio),FUERA DE META
7,IRAB,33.3%,< 2,PROXY: ejecutadas < programadas,FUERA DE META



KPIs calculados     : 8 de 8
Sobre datos reales  : 7  |  con proxy declarado: 1
Dentro de meta      : 2

Criterio README (minimo 7 KPIs funcionales): CUMPLE

Guardado: c:\Users\Home\Desktop\Grupo3\bigdata-g3-demanda-hospitalaria\data\gold\kpis_dashboard.parquet


In [4]:
# ============================================================
#  4. DATOS PARA LA VISTA 1 — DIRECTOR DE HOSPITAL
# ============================================================
# El README (seccion 4) define esta vista como: ocupacion actual vs
# prediccion de las proximas 4 semanas + alertas de pico. Se ensamblan
# dos insumos ya calculados en notebooks anteriores:
#   - prediccion_proximas_4_semanas (notebook 05)
#   - demanda_semanal_por_especialidad (para el nivel "actual")

# --- Prediccion de las proximas 4 semanas (tabla Gold del notebook 05) ---
pred = pd.read_parquet(RUTA_GOLD / "prediccion_proximas_4_semanas.parquet")
print(f"Prediccion cargada: {len(pred)} filas "
      f"({pred['especialidad'].nunique()} especialidades x "
      f"{pred['horizonte_semanas'].max()} semanas)\n")

# --- Demanda historica reciente por especialidad (nivel "actual") ---
dem = pd.read_parquet(RUTA_GOLD / "demanda_semanal_por_especialidad")
ult_semana = sorted(dem["semana"].unique())[-1]
actual = (dem[dem["semana"] == ult_semana]
          .groupby("especialidad")["total_atenciones"].sum()
          .reset_index(name="demanda_actual"))
print(f"Ultima semana observada: {ult_semana}\n")

# --- Media historica por especialidad (para medir el % de variacion) ---
media_hist = (dem.groupby("especialidad")["total_atenciones"]
              .mean().reset_index(name="media_historica"))

# --- Ensamblar: por especialidad, actual vs cada semana predicha ---
v1 = pred.merge(actual, on="especialidad", how="left") \
         .merge(media_hist, on="especialidad", how="left")
v1["variacion_vs_media_pct"] = ((v1["yhat"] - v1["media_historica"])
                                / v1["media_historica"] * 100).round(1)

# --- Alertas: variacion relevante respecto a la media historica ---
UMBRAL_ALERTA = 15  # % de desviacion que dispara alerta
def clasificar_alerta(pct):
    if pct >= UMBRAL_ALERTA:
        return "ALZA - preparar capacidad"
    if pct <= -UMBRAL_ALERTA:
        return "BAJA - posible subutilizacion"
    return "normal"
v1["alerta"] = v1["variacion_vs_media_pct"].apply(clasificar_alerta)

vista1 = v1[["semana", "especialidad", "horizonte_semanas", "demanda_actual",
             "yhat", "yhat_lower_80", "yhat_upper_80", "media_historica",
             "variacion_vs_media_pct", "alerta"]].copy()
vista1.columns = ["semana", "especialidad", "horizonte", "demanda_actual",
                  "prediccion", "pred_min_80", "pred_max_80", "media_historica",
                  "variacion_pct", "alerta"]
vista1 = vista1.round(1).sort_values(["especialidad", "horizonte"])

print("Vista 1 — ocupacion actual vs prediccion 4 semanas:\n")
display(vista1)

# --- Resumen de alertas ---
alertas = vista1[vista1["alerta"] != "normal"]
print(f"\nFilas con alerta: {len(alertas)} de {len(vista1)}")
if len(alertas):
    print("\nAlertas activas por especialidad:")
    for esp in alertas["especialidad"].unique():
        sub = alertas[alertas["especialidad"] == esp]
        print(f"   {esp}: {sub['alerta'].iloc[0]} "
              f"(hasta {sub['variacion_pct'].abs().max():.0f}% de desviacion)")

# --- Guardar para el dashboard ---
vista1.to_parquet(RUTA_GOLD / "vista1_director.parquet", index=False)
print(f"\nGuardado: {RUTA_GOLD / 'vista1_director.parquet'}")

Prediccion cargada: 16 filas (4 especialidades x 4 semanas)

Ultima semana observada: 2024-W52

Vista 1 — ocupacion actual vs prediccion 4 semanas:



,semana,especialidad,horizonte,demanda_actual,prediccion,pred_min_80,pred_max_80,media_historica,variacion_pct,alerta
0,2025-W01,Cardiologia,1,615,600.6,573.7,627.4,560.5,7.2,normal
1,2025-W02,Cardiologia,2,615,587.7,560.3,615.6,560.5,4.9,normal
2,2025-W03,Cardiologia,3,615,580.4,552.2,608.5,560.5,3.6,normal
3,2025-W04,Cardiologia,4,615,571.5,544.9,598.1,560.5,2.0,normal
4,2025-W01,Emergencia,1,797,904.7,844.1,963.3,558.8,61.9,ALZA - preparar capacidad
5,2025-W02,Emergencia,2,797,934.4,877.8,990.9,558.8,67.2,ALZA - preparar capacidad
6,2025-W03,Emergencia,3,797,953.8,893.3,1010.6,558.8,70.7,ALZA - preparar capacidad
7,2025-W04,Emergencia,4,797,977.4,919.0,1034.2,558.8,74.9,ALZA - preparar capacidad
8,2025-W01,Medicina Interna,1,1240,1176.0,1116.3,1238.0,1607.0,-26.8,BAJA - posible subutilizacion
9,2025-W02,Medicina Interna,2,1240,1173.3,1106.3,1232.7,1607.0,-27.0,BAJA - posible subutilizacion



Filas con alerta: 8 de 16

Alertas activas por especialidad:
   Emergencia: ALZA - preparar capacidad (hasta 75% de desviacion)
   Medicina Interna: BAJA - posible subutilizacion (hasta 28% de desviacion)

Guardado: c:\Users\Home\Desktop\Grupo3\bigdata-g3-demanda-hospitalaria\data\gold\vista1_director.parquet


In [5]:
# ============================================================
#  5. DATOS PARA LA VISTA 2 — GERENTE DE RED (MAPA DE LIMA)
# ============================================================
# El README (seccion 4) define esta vista como: mapa de Lima con
# establecimientos coloreados por demanda proyectada + redistribucion
# recomendada de recursos.
#
# Coordenadas: ubicaciones REALES de los 3 hospitales de EsSalud en Lima.
# Demanda por sede: la prediccion agregada de red (notebook 05) se reparte
# por participacion historica, con la misma logica de la celda 14 del
# notebook 05 (mantiene coherencia entre entregables).

# --- Coordenadas reales de los establecimientos ---
COORDENADAS = {
    "Hospital Almenara":   {"lat": -12.0576, "lon": -77.0179, "distrito": "La Victoria"},
    "Hospital Rebagliati": {"lat": -12.0797, "lon": -77.0428, "distrito": "Jesus Maria"},
    "Hospital Sabogal":    {"lat": -12.0589, "lon": -77.1120, "distrito": "Bellavista (Callao)"},
}

# --- Participacion historica por establecimiento ---
brechas = pd.read_parquet(RUTA_GOLD / "brechas_oferta_demanda_por_establecimiento")
share = (brechas.groupby("establecimiento_nombre")["demanda_atenciones"].sum())
share = (share / share.sum())

# --- Demanda predicha total de la red (horizonte 4 semanas) ---
pred = pd.read_parquet(RUTA_GOLD / "prediccion_proximas_4_semanas.parquet")
demanda_red_total = pred.groupby("horizonte_semanas")["yhat"].sum().mean()  # promedio 4 sem
print(f"Demanda predicha promedio de la red (por semana): {demanda_red_total:,.0f}\n")

# --- Oferta y capacidad por establecimiento (ultimo valor observado) ---
ult = brechas["semana"].max()
oferta = (brechas[brechas["semana"] == ult]
          .set_index("establecimiento_nombre"))

# --- Ensamblar la tabla del mapa ---
filas = []
for est, coord in COORDENADAS.items():
    part = share.get(est, 0)
    demanda_pred = demanda_red_total * part
    camas = oferta.loc[est, "camas_totales"]
    medicos = oferta.loc[est, "medicos_activos"]
    # oferta de atencion semanal (termino de flujo, coherente con notebook 05)
    oferta_sem = medicos * oferta.loc[est, "productividad_medica"]
    brecha = demanda_pred - oferta_sem
    filas.append({
        "establecimiento": est,
        "distrito": coord["distrito"],
        "lat": coord["lat"],
        "lon": coord["lon"],
        "participacion_pct": round(part * 100, 1),
        "demanda_predicha": round(demanda_pred, 0),
        "oferta_semanal": round(oferta_sem, 0),
        "brecha": round(brecha, 0),
        "camas_totales": int(camas),
        "medicos_activos": int(medicos),
    })

vista2 = pd.DataFrame(filas)

# --- Nivel de presion y recomendacion de redistribucion ---
def nivel(brecha):
    if brecha > 100:
        return "ALTA"
    if brecha > 0:
        return "MEDIA"
    return "HOLGADA"
vista2["nivel_presion"] = vista2["brecha"].apply(nivel)

# Redistribucion: de la sede mas holgada a la mas presionada
mas_presion = vista2.loc[vista2["brecha"].idxmax(), "establecimiento"]
mas_holgada = vista2.loc[vista2["brecha"].idxmin(), "establecimiento"]
vista2["recomendacion"] = vista2["establecimiento"].apply(
    lambda e: f"Recibir refuerzo desde {mas_holgada.split()[-1]}"
    if e == mas_presion else
    (f"Puede ceder recursos a {mas_presion.split()[-1]}"
     if e == mas_holgada else "Mantener"))

print("Vista 2 — demanda proyectada por establecimiento:\n")
display(vista2)

print(f"\nSede mas presionada: {mas_presion} "
      f"(brecha {vista2['brecha'].max():+.0f})")
print(f"Sede mas holgada   : {mas_holgada} "
      f"(brecha {vista2['brecha'].min():+.0f})")
print(f"Redistribucion sugerida: de {mas_holgada.split()[-1]} "
      f"hacia {mas_presion.split()[-1]}")

# --- Guardar para el dashboard ---
vista2.to_parquet(RUTA_GOLD / "vista2_mapa.parquet", index=False)
print(f"\nGuardado: {RUTA_GOLD / 'vista2_mapa.parquet'}")

Demanda predicha promedio de la red (por semana): 3,180

Vista 2 — demanda proyectada por establecimiento:



,establecimiento,distrito,lat,lon,participacion_pct,demanda_predicha,oferta_semanal,brecha,camas_totales,medicos_activos,nivel_presion,recomendacion
0,Hospital Almenara,La Victoria,-12.0576,-77.0179,33.3,1060.0,831.0,229.0,450,39,ALTA,Mantener
1,Hospital Rebagliati,Jesus Maria,-12.0797,-77.0428,33.3,1058.0,975.0,83.0,520,53,MEDIA,Puede ceder recursos a Sabogal
2,Hospital Sabogal,Bellavista (Callao),-12.0589,-77.1120,33.4,1061.0,741.0,320.0,380,38,ALTA,Recibir refuerzo desde Rebagliati



Sede mas presionada: Hospital Sabogal (brecha +320)
Sede mas holgada   : Hospital Rebagliati (brecha +83)
Redistribucion sugerida: de Rebagliati hacia Sabogal

Guardado: c:\Users\Home\Desktop\Grupo3\bigdata-g3-demanda-hospitalaria\data\gold\vista2_mapa.parquet


In [6]:
# ============================================================
#  6. DATOS PARA LA VISTA 3 — EPIDEMIOLOGO
# ============================================================
# El README (seccion 4) define esta vista como: correlacion
# clima-enfermedad + vigilancia de brotes + comparativo semana
# epidemiologica actual vs historico.
#
# Se ensamblan tres bloques:
#   A) Correlacion clima-enfermedad (tabla Gold del notebook 01)
#   B) Serie de dengue: sintetica (proyecto) vs real (CDC, notebook 06)
#   C) Contraste de la hipotesis H1 del README (rezago clima-dengue)
# El bloque B/C se reconstruye desde data/raw/ para que la celda sea
# autocontenida y no dependa del estado en memoria del notebook 06.

from scipy.stats import pearsonr

# ---------- A. Correlacion clima-enfermedad ----------
clima = pd.read_parquet(RUTA_GOLD / "correlacion_clima_enfermedad")
corr_vars = ["temperatura_max", "precipitacion_mm", "indice_humedad",
             "casos_dengue", "casos_influenza"]
corr_disp = [c for c in corr_vars if c in clima.columns]
matriz_corr = clima[corr_disp].corr().round(3)
print("A) Matriz de correlacion clima-enfermedad:\n")
display(matriz_corr)

# ---------- B. Serie sintetica vs real del CDC ----------
RUTA_RAW = raiz / "data" / "raw"
csv_cdc = RUTA_RAW / "minsa_cdc_vigilancia_dengue_2000_2024.csv"

# Serie sintetica (proyecto)
sint = clima[["semana", "casos_dengue"]].rename(
    columns={"casos_dengue": "dengue_sintetico"})

# Serie real (CDC, Lima 2022-2024) reconstruida desde el CSV
print("\nB) Reconstruyendo serie real del CDC desde data/raw/...")
cols = ["departamento", "ano", "semana"]
trozos = []
for bloque in pd.read_csv(csv_cdc, sep=";", encoding="utf-8-sig",
                          usecols=cols, chunksize=200_000, low_memory=False):
    bloque["departamento"] = bloque["departamento"].astype(str).str.strip().str.upper()
    sel = bloque[(bloque["departamento"] == "LIMA") & (bloque["ano"].isin([2022,2023,2024]))]
    if len(sel):
        trozos.append(sel)
cdc = pd.concat(trozos, ignore_index=True)
cdc_sem = cdc.groupby(["ano","semana"]).size().reset_index(name="dengue_real")
cdc_sem["clave"] = (cdc_sem["ano"].astype(str) + "-W"
                    + cdc_sem["semana"].astype(int).astype(str).str.zfill(2))

# Union con calendario completo (ceros explicitos)
vista3 = sint.rename(columns={"semana": "clave"}).merge(
    cdc_sem[["clave", "dengue_real"]], on="clave", how="left")
vista3["dengue_real"] = vista3["dengue_real"].fillna(0)
vista3["anio"] = vista3["clave"].str[:4]
print(f"   Semanas en la serie: {len(vista3)}")
print(f"   Casos reales (Lima 2022-2024): {int(vista3['dengue_real'].sum()):,}")
print(f"   Casos sinteticos            : {int(vista3['dengue_sintetico'].sum()):,}\n")

# ---------- C. Comparativo por anio ----------
comp_anual = (vista3.groupby("anio")
              .agg(real=("dengue_real","sum"),
                   sintetico=("dengue_sintetico","sum"))
              .reset_index())
comp_anual["ratio_real_sint"] = (comp_anual["real"]/comp_anual["sintetico"]).round(2)
print("C) Comparativo anual real vs sintetico:\n")
display(comp_anual)

crec_real = comp_anual["real"].iloc[-1] / comp_anual["real"].iloc[0]
crec_sint = comp_anual["sintetico"].iloc[-1] / comp_anual["sintetico"].iloc[0]
print(f"\n   Crecimiento 2022->2024  real: {crec_real:.1f}x  |  "
      f"sintetico: {crec_sint:.2f}x")
print("   -> El generador no reproduce la dinamica epidemica real.")
print("      Confirma la limitacion declarada en el notebook 05.\n")

# --- Guardar los tres insumos para el dashboard ---
matriz_corr.to_parquet(RUTA_GOLD / "vista3_correlacion.parquet")
vista3.to_parquet(RUTA_GOLD / "vista3_series_dengue.parquet", index=False)
comp_anual.to_parquet(RUTA_GOLD / "vista3_comparativo_anual.parquet", index=False)
print(f"Guardado: vista3_correlacion, vista3_series_dengue, "
      f"vista3_comparativo_anual (en data/gold/)")

A) Matriz de correlacion clima-enfermedad:



,temperatura_max,precipitacion_mm,indice_humedad,casos_dengue,casos_influenza
temperatura_max,1.000,-0.865,-0.222,0.895,-0.814
precipitacion_mm,-0.865,1.000,0.187,-0.758,0.860
indice_humedad,-0.222,0.187,1.000,-0.232,0.160
casos_dengue,0.895,-0.758,-0.232,1.000,-0.774
casos_influenza,-0.814,0.860,0.160,-0.774,1.000



B) Reconstruyendo serie real del CDC desde data/raw/...
   Semanas en la serie: 156
   Casos reales (Lima 2022-2024): 119,537
   Casos sinteticos            : 22,123

C) Comparativo anual real vs sintetico:



,anio,real,sintetico,ratio_real_sint
0,2022,938.0,6813.0,0.14
1,2023,30735.0,8042.0,3.82
2,2024,87864.0,7268.0,12.09



   Crecimiento 2022->2024  real: 93.7x  |  sintetico: 1.07x
   -> El generador no reproduce la dinamica epidemica real.
      Confirma la limitacion declarada en el notebook 05.

Guardado: vista3_correlacion, vista3_series_dengue, vista3_comparativo_anual (en data/gold/)


In [7]:
# ============================================================
#  7. VERIFICACION DE INSUMOS PARA EL DASHBOARD
# ============================================================
# La app Streamlit leera estos archivos de data/gold/. Se verifica que
# todos existan y sean legibles antes de construir el dashboard, para que
# la app no falle al abrir.

ARCHIVOS_DASHBOARD = {
    "kpis_dashboard.parquet":          "KPIs de gestion (8)",
    "vista1_director.parquet":         "Vista 1 - ocupacion vs prediccion",
    "vista2_mapa.parquet":             "Vista 2 - mapa de Lima",
    "vista3_correlacion.parquet":      "Vista 3 - matriz correlacion",
    "vista3_series_dengue.parquet":    "Vista 3 - series dengue real vs sint.",
    "vista3_comparativo_anual.parquet":"Vista 3 - comparativo anual",
    "prediccion_proximas_4_semanas.parquet": "Prediccion 4 semanas (nb 05)",
}

print("Verificacion de insumos del dashboard:\n")
filas = []
todo_ok = True
for archivo, desc in ARCHIVOS_DASHBOARD.items():
    ruta = RUTA_GOLD / archivo
    if ruta.exists():
        try:
            df = pd.read_parquet(ruta)
            filas.append({"archivo": archivo, "descripcion": desc,
                          "filas": len(df), "columnas": df.shape[1],
                          "estado": "OK"})
        except Exception as e:
            filas.append({"archivo": archivo, "descripcion": desc,
                          "filas": 0, "columnas": 0,
                          "estado": f"ERROR: {type(e).__name__}"})
            todo_ok = False
    else:
        filas.append({"archivo": archivo, "descripcion": desc,
                      "filas": 0, "columnas": 0, "estado": "FALTA"})
        todo_ok = False

verif = pd.DataFrame(filas)
display(verif)

print(f"\nArchivos verificados: {len(verif)}")
print(f"Todos disponibles   : {'SI' if todo_ok else 'NO -> revisar los marcados'}")
print(f"\nEstado para construir el dashboard: "
      f"{'LISTO' if todo_ok else 'INCOMPLETO'}")

Verificacion de insumos del dashboard:



,archivo,descripcion,filas,columnas,estado
0,kpis_dashboard.parquet,KPIs de gestion (8),8,5,OK
1,vista1_director.parquet,Vista 1 - ocupacion vs prediccion,16,10,OK
2,vista2_mapa.parquet,Vista 2 - mapa de Lima,3,12,OK
3,vista3_correlacion.parquet,Vista 3 - matriz correlacion,5,5,OK
4,vista3_series_dengue.parquet,Vista 3 - series dengue real vs sint.,156,4,OK
5,vista3_comparativo_anual.parquet,Vista 3 - comparativo anual,3,4,OK
6,prediccion_proximas_4_semanas.parquet,Prediccion 4 semanas (nb 05),16,12,OK



Archivos verificados: 7
Todos disponibles   : SI

Estado para construir el dashboard: LISTO


In [8]:
# ============================================================
#  8. CONCLUSIONES — NOTEBOOK 07 (CAPA DE DATOS DEL DASHBOARD)
# ============================================================

L = "=" * 66
print(L); print("CONCLUSIONES — NOTEBOOK 07 (SEMANA 8)"); print(L)

# ---------- 1. KPIs ----------
n_real = int(kpi_df["fuente"].str.startswith("real").sum())
print(f"""
1) KPIs DE GESTION HOSPITALARIA (README seccion 7.4)

   Calculados : {len(kpi_df)} de 8  ->  cumple el minimo de 7 del checklist
   Datos reales: {n_real}  |  proxy declarado: {len(kpi_df) - n_real}

   Precisiones metodologicas declaradas:
   - Promedio de Estadia se calcula solo sobre hospitalizaciones.
   - Rendimiento Medico usa 'medicos_activos' (121 en total), no los
     9,000 medico_id del HIS (identificadores sinteticos repartidos por
     igual entre las 3 sedes: quinto caso de campo incoherente).
   - IRAB es un proxy: deficit entre consultas ejecutadas y programadas.
   - Mortalidad es correcta como calculo pero los desenlaces del dataset
     son aleatorios, por lo que no es clinicamente interpretable.

   La mayoria de los KPIs queda FUERA de meta EsSalud. Es lo esperado:
   se mide un hospital sintetico, no la gestion real. El valor del tablero
   es que DETECTARIA esos desvios si operara sobre datos reales.
""")

# ---------- 2. Las tres vistas ----------
print(L); print("2) VISTAS DEL DASHBOARD (README seccion 4)"); print(L)
print(f"""
   Vista 1 — Director de Hospital
      Ocupacion actual vs prediccion 4 semanas, con alertas automaticas.
      Emergencia dispara ALZA (+62% a +75%, pico dengue de enero) y
      Medicina Interna BAJA (-27% a -28%). Coincide con la estacionalidad
      inversa validada en el notebook 01.

   Vista 2 — Gerente de Red (mapa de Lima)
      Los 3 hospitales geolocalizados con coordenadas reales, coloreados
      por presion de demanda. Sabogal es el mas presionado (menor
      dotacion: 38 medicos); se recomienda refuerzo desde Rebagliati.

   Vista 3 — Epidemiologo
      Correlacion clima-enfermedad (temperatura-dengue +0.895,
      temperatura-influenza -0.814), y contraste del generador sintetico
      contra la vigilancia REAL del CDC: crecimiento real 93.7x vs 1.07x
      del generador entre 2022 y 2024.
""")

# ---------- 3. Cumplimiento del checklist ----------
print(L); print("3) CHECKLIST DE ENTREGA FINAL (README seccion 15)"); print(L)
chk = pd.DataFrame([
    {"requisito": "Dashboard con minimo 7 KPIs funcionales",
     "estado": "CUMPLE" if len(kpi_df) >= 7 else "NO CUMPLE"},
    {"requisito": "Mapa Lima con demanda por establecimiento",
     "estado": "CUMPLE"},
    {"requisito": "Insumos de las 3 vistas persistidos en Gold",
     "estado": "CUMPLE"},
])
display(chk)

# ---------- 4. Arquitectura del dashboard ----------
print(L); print("4) ARQUITECTURA DEL DASHBOARD"); print(L)
print("""
   Este notebook es la CAPA DE DATOS: calcula, valida y persiste en
   data/gold/ los 7 insumos que consume la app. La CAPA DE PRESENTACION
   es una app Streamlit (src/dashboard.py) que lee esos archivos y
   renderiza las 3 vistas.

   El dashboard se ejecuta localmente con 'streamlit run'. No se despliega
   en Streamlit Cloud porque los datos (capa Gold, 500K atenciones) estan
   excluidos del repositorio por .gitignore y superan el limite de GitHub.
   La demo en vivo se realiza en local, como especifica el README (8).
""")

print(L); print("5) PENDIENTES DE LA EF"); print(L)
print("""
   - App Streamlit (src/dashboard.py) con las 3 vistas
   - docs/arquitectura_demanda_hospitalaria.png
   - Abstract Scopus (250 palabras, ingles)
   - README actualizado con resultados reales del Grupo 3
   - Pull Request de la Evaluacion Final
   - Tecnicos: requirements.txt (prophet 1.3.0, great-expectations 0.18.15,
     streamlit, folium, beautifulsoup4); HADOOP_HOME hardcodeado
""")
print(L)

CONCLUSIONES — NOTEBOOK 07 (SEMANA 8)

1) KPIs DE GESTION HOSPITALARIA (README seccion 7.4)

   Calculados : 8 de 8  ->  cumple el minimo de 7 del checklist
   Datos reales: 7  |  proxy declarado: 1

   Precisiones metodologicas declaradas:
   - Promedio de Estadia se calcula solo sobre hospitalizaciones.
   - Rendimiento Medico usa 'medicos_activos' (121 en total), no los
     9,000 medico_id del HIS (identificadores sinteticos repartidos por
     igual entre las 3 sedes: quinto caso de campo incoherente).
   - IRAB es un proxy: deficit entre consultas ejecutadas y programadas.
   - Mortalidad es correcta como calculo pero los desenlaces del dataset
     son aleatorios, por lo que no es clinicamente interpretable.

   La mayoria de los KPIs queda FUERA de meta EsSalud. Es lo esperado:
   se mide un hospital sintetico, no la gestion real. El valor del tablero
   es que DETECTARIA esos desvios si operara sobre datos reales.

2) VISTAS DEL DASHBOARD (README seccion 4)

   Vista 1 — Direc

,requisito,estado
0,Dashboard con minimo 7 KPIs funcionales,CUMPLE
1,Mapa Lima con demanda por establecimiento,CUMPLE
2,Insumos de las 3 vistas persistidos en Gold,CUMPLE


4) ARQUITECTURA DEL DASHBOARD

   Este notebook es la CAPA DE DATOS: calcula, valida y persiste en
   data/gold/ los 7 insumos que consume la app. La CAPA DE PRESENTACION
   es una app Streamlit (src/dashboard.py) que lee esos archivos y
   renderiza las 3 vistas.

   El dashboard se ejecuta localmente con 'streamlit run'. No se despliega
   en Streamlit Cloud porque los datos (capa Gold, 500K atenciones) estan
   excluidos del repositorio por .gitignore y superan el limite de GitHub.
   La demo en vivo se realiza en local, como especifica el README (8).

5) PENDIENTES DE LA EF

   - App Streamlit (src/dashboard.py) con las 3 vistas
   - docs/arquitectura_demanda_hospitalaria.png
   - Abstract Scopus (250 palabras, ingles)
   - README actualizado con resultados reales del Grupo 3
   - Pull Request de la Evaluacion Final
   - Tecnicos: requirements.txt (prophet 1.3.0, great-expectations 0.18.15,
     streamlit, folium, beautifulsoup4); HADOOP_HOME hardcodeado

